# Khám phá dữ liệu Abalone

Notebook này dùng để phân tích dữ liệu ban đầu cho bài toán dự đoán `Rings` của bộ dữ liệu Abalone.

## 1. Chuẩn bị thư viện

In [2]:
import warnings # dùng để bỏ qua các cảnh báo không cần thiết
warnings.filterwarnings('ignore') # bỏ qua tất cả các cảnh báo

from pathlib import Path # thư viện để làm việc với đường dẫn tệp

import numpy as np # thư viện để làm việc với mảng và tính toán số học
import pandas as pd # thư viện để làm việc với dữ liệu dạng bảng
import matplotlib.pyplot as plt # thư viện để vẽ biểu đồ
import seaborn as sns # thư viện để vẽ biểu đồ đẹp hơn

sns.set_theme(style='whitegrid', palette='deep') # thiết lập chủ đề cho biểu đồ
plt.rcParams['figure.figsize'] = (10, 6) # thiết lập kích thước mặc định cho biểu đồ

## 2. Đọc dữ liệu

In [3]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve() 
# xác định thư mục gốc của dự án, nếu đang ở trong thư mục 'notebooks' thì lấy thư mục cha, ngược lại lấy chính thư mục hiện tại


# Đường dẫn đến tệp dữ liệu, được xây dựng dựa trên thư mục gốc của dự án
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'abalone.csv'

columns = [
    'Sex',
    'Length',
    'Diameter',
    'Height',
    'WholeWeight',
    'ShuckedWeight',
    'VisceraWeight',
    'ShellWeight',
    'Rings',
]

# Đọc dữ liệu từ tệp CSV vào một DataFrame của pandas, sử dụng các tên cột đã định nghĩa ở trên
df = pd.read_csv(DATA_PATH, header=None, names=columns)
df.head() # hiển thị 5 dòng đầu tiên của DataFrame để kiểm tra dữ liệu đã được đọc đúng chưa

,Sex,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


## 3. Thông tin tổng quan

In [ ]:
print(f'Kích thước dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột')
display(df.info())
display(df.describe(include='all').T)

## 4. Kiểm tra giá trị thiếu và trùng lặp

In [ ]:
missing_summary = df.isna().sum().to_frame('missing_count')
missing_summary['missing_ratio'] = missing_summary['missing_count'] / len(df)
display(missing_summary)

print('Số dòng trùng lặp:', df.duplicated().sum())

In [ ]:
missing_summary = df.isna().sum().to_frame('missing_count') # tính tổng số giá trị thiếu cho mỗi cột và lưu vào một DataFrame mới có tên 'missing_summary'
missing_summary['missing_ratio'] = missing_summary['missing_count'] / len(df) # tính tỷ lệ phần trăm giá trị thiếu cho mỗi cột và thêm vào DataFrame 'missing_summary' dưới dạng một cột mới có tên 'missing_ratio'
display(missing_summary) # hiển thị DataFrame 'missing_summary' để xem số lượng và tỷ lệ phần trăm giá trị thiếu cho mỗi cột

print('Số dòng trùng lặp:', df.duplicated().sum()) 
# tính và in ra số lượng dòng trùng lặp trong DataFrame 'df' bằng cách sử dụng phương thức 'duplicated()' của pandas, sau đó tính tổng số dòng trùng lặp bằng cách sử dụng phương thức 'sum()' và in kết

## 5. Phân tích biến phân loại `Sex`

In [ ]:
sex_counts = df['Sex'].value_counts()
display(sex_counts)

# Vẽ biểu đồ cột để hiển thị phân phối của biến 'Sex' trong DataFrame 'df' bằng cách sử dụng thư viện seaborn. 
# Biểu đồ sẽ được sắp xếp theo thứ tự của các giá trị trong 'sex_counts' để đảm bảo rằng các cột được hiển thị theo đúng thứ tự mong muốn.
ax = sns.countplot(data=df, x='Sex', order=sex_counts.index)
ax.set_title('Phân phối biến Sex')
plt.show()

## 6. Phân phối biến mục tiêu `Rings`

In [ ]:
display(df['Rings'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Vẽ biểu đồ histogram kết hợp với đường mật độ (kde) cho biến 'Rings' trên trục đầu tiên (axes[0])
# vẽ biểu đồ boxplot cho biến 'Rings' trên trục thứ hai (axes[1]).
sns.histplot(df['Rings'], kde=True, ax=axes[0])
axes[0].set_title('Histogram của Rings')
sns.boxplot(x=df['Rings'], ax=axes[1])
axes[1].set_title('Boxplot của Rings')
# Điều chỉnh bố cục của biểu đồ để tránh chồng lấn giữa các phần tử và đảm bảo rằng tất cả các tiêu đề và nhãn đều hiển thị rõ ràng.
plt.tight_layout()
plt.show()

## 7. Phân tích các biến số

In [ ]:
# Chọn các cột có kiểu dữ liệu số để vẽ biểu đồ histogram cho tất cả các biến số trong DataFrame 'df'. 
# \Biểu đồ sẽ có 20 bins và kích thước tổng thể là 16x12 inch. 
# Sau đó, tiêu đề chung cho tất cả các biểu đồ sẽ được đặt là 'Phân phối các biến số' và bố cục sẽ được điều chỉnh để tránh chồng lấn giữa các phần tử.
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols].hist(bins=20, figsize=(16, 12))
plt.suptitle('Phân phối các biến số', y=1.02)
plt.tight_layout()
plt.show()

## 8. Tương quan giữa các biến số

In [ ]:
corr = df[numeric_cols].corr()
# Hiển thị hệ số tương quan giữa các biến số với biến 'Rings', được sắp xếp theo thứ tự giảm dần để dễ dàng nhận biết những biến nào có mối quan hệ mạnh nhất với 'Rings'.
display(corr['Rings'].sort_values(ascending=False))
# Vẽ biểu đồ heatmap để hiển thị ma trận tương quan giữa các biến số trong DataFrame 'df' sử dụng thư viện seaborn. 
# Biểu đồ sẽ có kích thước 10x8 inch, sử dụng bảng màu 'coolwarm' để thể hiện mức độ tương quan, và hiển thị giá trị tương quan với định dạng 2 chữ số thập phân. 
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Ma trận tương quan')
plt.show()

## 9. Quan hệ giữa đặc trưng và biến mục tiêu

In [ ]:
# Dựa trên hệ số tương quan đã tính ở trên, chọn ra các biến có mối quan hệ mạnh nhất với biến 'Rings' để vẽ biểu đồ scatter plot 
key_features = ['Length', 'Diameter', 'Height', 'WholeWeight', 'ShellWeight']

# Vẽ biểu đồ scatter plot để hiển thị mối quan hệ giữa các biến đã chọn trong 'key_features' với biến 'Rings'
fig, axes = plt.subplots(len(key_features), 1, figsize=(8, 4 * len(key_features)))

# Sử dụng vòng lặp để vẽ biểu đồ scatter plot cho từng biến trong 'key_features' 
for ax, col in zip(axes, key_features):
    sns.scatterplot(data=df, x=col, y='Rings', hue='Sex', alpha=0.6, ax=ax)
    ax.set_title(f'{col} và Rings')
plt.tight_layout()
plt.show()

## 10. Kiểm tra ngoại lệ

In [ ]:
#fig, axes dùng để tạo một lưới các biểu đồ, trong đó 'fig' là đối tượng hình ảnh tổng thể và 'axes' là một mảng các trục (axes) tương ứng với từng biểu đồ.
fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(10, 4 * len(numeric_cols)))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(f'Boxplot của {col}')
plt.tight_layout()
plt.show()

## 11. Nhận xét ban đầu

In [ ]:
print('1. Dữ liệu không có giá trị thiếu.')
print('2. Bài toán phù hợp với hồi quy, biến mục tiêu là Rings.')
print('3. Sex là biến phân loại cần mã hóa ở bước tiền xử lý.')
print('4. Cần kiểm tra kỹ ngoại lệ và mức độ lệch phân phối của các biến trọng lượng.')
print('5. Nên thử cả mô hình tuyến tính và mô hình cây/ensemble ở giai đoạn baseline.')